# LeNet-5

## 0. 基本认识 LeNet-5 Overview

LeNet-5 是早期经典的卷积神经网络 CNN，主要用于手写数字识别。课件中给出的背景是：

- 发明者：贝尔实验室研究员 Yann LeCun。
- 提出时间：1989 年左右。
- 研究目的：完成手写数字识别。

LeNet-5 把图像分类过程拆成两个阶段：

- 特征提取 Feature Extraction：用卷积层和池化层从图像中提取局部特征。
- 分类 Classification：把特征展平后，交给全连接层输出类别分数。

核心思想是：卷积层负责识别局部模式，池化层负责降低特征图尺寸，全连接层负责完成最终分类。

---

## 1. 网络结构 Network Architecture

### 1.1 两部分组成

| 部分 | 包含层 | 作用 |
| --- | --- | --- |
| 特征提取器 | 2 个卷积层 + 2 个平均池化层 | 从图像中提取边缘、纹理等局部特征 |
| 分类器 | 3 个全连接层 | 根据提取到的特征输出类别分数 |

### 1.2 课件中的尺寸变化

课件和配套源码使用的是 $28 \times 28 \times 1$ 灰度图输入。第一层卷积使用 padding=2，第二层卷积使用 padding=0：

| 顺序 | 层 | 主要参数 | 输入尺寸 | 输出尺寸 |
| --- | --- | --- | --- | --- |
| 1 | Input | 灰度图 | - | $1 \times 28 \times 28$ |
| 2 | Conv1 | $5 \times 5$，6 个卷积核，stride=1，padding=2 | $1 \times 28 \times 28$ | $6 \times 28 \times 28$ |
| 3 | AvgPool1 | $2 \times 2$，stride=2 | $6 \times 28 \times 28$ | $6 \times 14 \times 14$ |
| 4 | Conv2 | $5 \times 5$，16 个卷积核，stride=1，padding=0 | $6 \times 14 \times 14$ | $16 \times 10 \times 10$ |
| 5 | AvgPool2 | $2 \times 2$，stride=2 | $16 \times 10 \times 10$ | $16 \times 5 \times 5$ |
| 6 | Flatten | 展平 | $16 \times 5 \times 5$ | $400$ |
| 7 | FC1 | Linear | $400$ | $120$ |
| 8 | FC2 | Linear | $120$ | $84$ |
| 9 | FC3 | Linear | $84$ | $10$ |

第一层卷积加 padding=2 后，$28 \times 28$ 的空间尺寸保持不变；第二层卷积不加 padding，所以 $14 \times 14$ 会变成 $10 \times 10$。

### 1.3 模型单元结构

LeNet-5 的基础单元可以理解为：

```text
Convolution -> Activation -> Average Pooling
```

- 课件中强调的是 `卷积层 + sigmoid 激活函数 + 平均池化层`。
- 配套源码里使用 `nn.Sigmoid()` 和 `nn.AvgPool2d()`，和课件描述一致。
- 最后一层输出 10 个分类，对应 10 个类别。

### 1.4 数据维度

PyTorch 中卷积层常用的数据格式是 `(B, C, H, W)`：

- $B$ Batch：批大小，一次输入网络的图片数量。
- $C$ Channel：通道数。灰度图为 1，RGB 图为 3；卷积输出中表示特征图数量。
- $H$ Height：特征图高度。
- $W$ Width：特征图宽度。

卷积层输出可以记作 `(B, FN, OH, OW)`：

- $FN$ Filter Number：卷积核数量，也就是输出特征图数量。
- $OH$ Output Height：输出特征图高度。
- $OW$ Output Width：输出特征图宽度。

全连接层输入前需要先展平，课件对应的维度变化是：

```text
(B, 16, 5, 5) -> (B, 400) -> (B, out_features)
```

### 1.5 注意点

- 课件写的是 `(B, C, W, H)`，实际 PyTorch 中常用顺序是 `(B, C, H, W)`。
- 课件中的第一层卷积有 padding=2，所以这里的 Flatten 是 400；如果不加 padding，Flatten 会变成 256。
- `FL` 在课件里表示全连接层输出长度，但代码中更常见的名字是 `out_features`。
- 画网络结构图时，可以按 `输入 -> 卷积 -> 池化 -> 卷积 -> 池化 -> 展平 -> 全连接 -> 输出` 这一条主线画。

---

## 2. 网络参数计算 Parameter Calculation

### 2.1 卷积输出尺寸

卷积层输出的空间尺寸一般按下面公式计算：

$$
O = \frac{I + 2P - K}{S} + 1
$$

其中：

- $I$：输入尺寸。
- $P$：padding 大小。
- $K$：卷积核大小。
- $S$：stride 步幅。
- $O$：输出尺寸。

### 2.2 两个卷积层的尺寸计算

第一层卷积：输入 $28 \times 28$，卷积核 $5 \times 5$，padding=2，stride=1。

$$
O = \frac{28 + 2 \times 2 - 5}{1} + 1 = 28
$$

所以输出为 $6 \times 28 \times 28$。

第二层卷积：输入 $14 \times 14$，卷积核 $5 \times 5$，padding=0，stride=1。

$$
O = \frac{14 + 2 \times 0 - 5}{1} + 1 = 10
$$

所以输出为 $16 \times 10 \times 10$。

### 2.3 池化和展平

平均池化层使用 $2 \times 2$ 感受野，stride=2，会把宽高各缩小一半：

```text
28 x 28 -> 14 x 14
10 x 10 -> 5 x 5
```

第二次池化后得到 $16 \times 5 \times 5$，展平后长度为：

$$
16 \times 5 \times 5 = 400
$$

这就是源码中 `nn.Linear(400, 120)` 的原因。

---

## 3. PyTorch 实现 PyTorch Implementation

### 3.1 模型结构

配套源码中的 LeNet 模型可以概括为：

```python
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.c1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)
        self.sig = nn.Sigmoid()
        self.s2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.c3 = nn.Conv2d(6, 16, kernel_size=5)
        self.s4 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.f5 = nn.Linear(400, 120)
        self.f6 = nn.Linear(120, 84)
        self.f7 = nn.Linear(84, 10)
```

前向传播顺序：

```text
Conv1 -> Sigmoid -> AvgPool1 -> Conv2 -> Sigmoid -> AvgPool2 -> Flatten -> FC1 -> FC2 -> FC3
```

### 3.2 实战数据集

源码使用 `torchvision.datasets.FashionMNIST`：

- 训练集：`train=True`。
- 测试集：`train=False`。
- 图像预处理：`transforms.Resize(size=28)` 和 `transforms.ToTensor()`。
- 输入张量形状：`(B, 1, 28, 28)`。
- 分类数量：10 类。

### 3.3 训练流程

训练代码的大致流程是：

1. 读取 FashionMNIST，并按 8:2 划分训练集和验证集。
2. 使用 `DataLoader` 按 batch 读取数据。
3. 构建 `LeNet()` 模型，并把模型放到 GPU 或 CPU。
4. 使用 Adam 优化器，学习率为 0.001。
5. 使用交叉熵损失 `nn.CrossEntropyLoss()`。
6. 每个 epoch 中先训练，再验证，记录 loss 和 accuracy。
7. 保存验证集准确率最高的模型参数。

### 3.4 测试流程

测试代码的流程是：

1. 加载 `best_model.pth` 中保存的模型参数。
2. 读取 FashionMNIST 测试集。
3. 使用 `torch.no_grad()` 关闭梯度计算。
4. 前向传播得到输出。
5. 使用 `torch.argmax(output, dim=1)` 得到预测类别。
6. 统计预测正确数量，计算测试准确率。

注意：模型最后一层直接输出 logits，训练时交给 `CrossEntropyLoss` 即可，不需要在模型中手动加 `Softmax`。

---

## 4. 总结 Summary

- LeNet-5 是最早发布并产生重要影响的卷积神经网络之一。
- 它由卷积层、非线性激活函数、平均池化层和全连接层组成。
- 卷积层逐渐提取更高层的局部特征，池化层降低空间分辨率，全连接层完成分类。
- 课件和源码采用 $1 \times 28 \times 28$ 输入，第一层卷积 padding=2，所以展平长度是 400。
- 传统 LeNet 使用 sigmoid 和平均池化；后续更深的 CNN 常用 ReLU、最大池化、Dropout 等改进。

***********